# Notebook 2 — Construcción de Trayectorias Académicas y Variable de Permanencia
**Proyecto:** Academic Performance, Course Load, and Student Persistence  
**Institución:** Instituto Tecnológico Metropolitano (ITM)  

---
## Objetivo
Transformar los 7 archivos limpios (uno por semestre) en un dataset longitudinal
donde cada fila representa la trayectoria completa de un estudiante en una materia.
Se define además la variable binaria de **Permanencia Estudiantil**.

## Flujo
```
ETL _process/Cleaning_Data/*_ETLclean.xlsx
        │
        ▼  Normalización + codificación binaria de Resultado
        │  Unificación cronológica de semestres
        │  Historial por estudiante-materia
        │  Cálculo y corrección de Permanencia
        ▼
Analytical_Dataset_Building/Study_Data/Oficial_Data_ToAnalisis.xlsx
```

## 1. Librerías y configuración de rutas

In [1]:
import ast
import glob
import os
import numpy as np
import pandas as pd
from pathlib import Path

# ── Rutas ─────────────────────────────────────────────────────────────────────
# El notebook vive en:  Proyecto_Final_STEM/Analytical_Dataset_Building/
# La raíz del proyecto: Proyecto_Final_STEM/
NOTEBOOK_DIR   = Path().resolve()                       # Analytical_Dataset_Building/
PROJECT_ROOT   = NOTEBOOK_DIR.parent                    # Proyecto_Final_STEM/

CARPETA_LIMPIA = PROJECT_ROOT / 'ETL _process' / 'Cleaning_Data'
CARPETA_SALIDA = NOTEBOOK_DIR / 'Study_Data'
RUTA_SALIDA    = CARPETA_SALIDA / 'Oficial_Data_ToAnalisis.xlsx'

print(f'Entrada : {CARPETA_LIMPIA}')
print(f'Salida  : {RUTA_SALIDA}')
assert CARPETA_LIMPIA.exists(), f'Carpeta de entrada no encontrada: {CARPETA_LIMPIA}'


Entrada : /sessions/trusting-laughing-brahmagupta/mnt/Academic Performance, Course Load, and Student Persistence A Longitudinal Modeling of Academic Trajectories in an Engineering Program/Proyecto_Final_STEM_COPIA/ETL _process/Cleaning_Data
Salida  : /sessions/trusting-laughing-brahmagupta/mnt/Academic Performance, Course Load, and Student Persistence A Longitudinal Modeling of Academic Trajectories in an Engineering Program/Proyecto_Final_STEM_COPIA/Analytical_Dataset_Building/Study_Data/Oficial_Data_ToAnalisis.xlsx


## 2. Carga y validación de los datos limpios
Se carga cada archivo `*_ETLclean.xlsx` en un diccionario `{semestre: DataFrame}`
y se validan los valores únicos de la columna `Resultado` para detectar
inconsistencias entre semestres.

In [2]:
archivos = sorted(CARPETA_LIMPIA.glob('*_ETLclean.xlsx'))
if not archivos:
    raise FileNotFoundError(f'No se encontraron archivos ETLclean en {CARPETA_LIMPIA}')

dta_frames = {}
for archivo in archivos:
    nombre = archivo.name.replace('_ETLclean.xlsx', '')
    dta_frames[nombre] = pd.read_excel(archivo)

print(f'Semestres cargados ({len(dta_frames)}): {sorted(dta_frames.keys())}\n')

print('Valores únicos de Resultado por semestre:')
for nombre, df in sorted(dta_frames.items()):
    print(f'  {nombre}: {sorted(df["Resultado"].dropna().unique().tolist())}')


Semestres cargados (7): ['2022-1', '2022-2', '2023-1', '2023-2', '2024-1', '2024-2', '2025-1']

Valores únicos de Resultado por semestre:
  2022-1: ['Aprobó', 'Canceló', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó']
  2022-2: ['Aprobó', 'Canceló', 'Condicion Especial', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']
  2023-1: ['Aprobó', 'Canceló', 'Evaluación Incompleta', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']
  2023-2: ['Aprobó', 'Canceló', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']
  2024-1: ['Aprobó', 'Canceló', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']
  2024-2: ['Aprobó', 'Canceló', 'Evaluación Incompleta', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']
  2025-1: ['Aprobó', 'Canceló', 'Habilitó y Aprobó', 'Habilitó y Reprobó', 'Reprobó', 'Reprobó Inasistencia']


## 3. Normalización y codificación binaria de resultados
Se estandarizan los textos de las columnas clave y se mapea `Resultado`
a una variable numérica binaria (`Resultado_num`):
- **1** → aprobó la materia (Aprobó / Habilitó y Aprobó)
- **0** → cualquier otro resultado (Reprobó, Canceló, Reprobó Inasistencia, etc.)

In [3]:
# Diccionario de mapeo: texto → binario
MAPA_RESULTADOS = {
    'Aprobó'               : 1,
    'Habilitó y Aprobó'    : 1,
    'Reprobó'              : 0,
    'Habilitó y Reprobó'   : 0,
    'Canceló'              : 0,
    'Reprobó Inasistencia' : 0,
    'Condicion Especial'   : 0,
    'Evaluación Incompleta': 0,
}

for sem_label, df in dta_frames.items():
    df['Semestre']      = sem_label
    df['Documento']     = df['Documento'].astype(str).str.strip()
    df['Materia']       = df['Materia'].astype(str).str.strip()
    df['Resultado']     = df['Resultado'].astype(str).str.strip()
    df['Resultado_num'] = df['Resultado'].map(MAPA_RESULTADOS).fillna(0).astype(int)

print('Codificación aplicada correctamente.')


Codificación aplicada correctamente.


## 4. Unificación en dataset longitudinal
Se concatenan todos los semestres y se crea `Orden_Cronologico` (entero
en formato YYYYS, e.g. 20221) para ordenar registros en el tiempo.

In [4]:
def sem_to_order(sem) -> int:
    """
    Convierte un identificador de semestre al orden cronológico entero YYYYS.

    Acepta tanto strings ('2022-1') como enteros (20221) o floats (20221.0).

    Examples
    --------
    >>> sem_to_order('2022-1')
    20221
    >>> sem_to_order(20222)
    20222
    """
    if isinstance(sem, (int, np.integer)):
        return int(sem)
    try:
        year, term = str(sem).split('-')
        return int(year) * 10 + int(term)
    except ValueError:
        try:
            return int(float(sem))
        except Exception:
            return None


df_all = pd.concat(dta_frames.values(), ignore_index=True)
df_all['Orden_Cronologico'] = df_all['Semestre'].apply(sem_to_order)
df_all = df_all.sort_values(['Documento', 'Materia', 'Orden_Cronologico']).reset_index(drop=True)

print(f'Registros totales: {len(df_all):,}')
print(f'Estudiantes únicos: {df_all["Documento"].nunique():,}')
print(f'Semestres: {sorted(df_all["Orden_Cronologico"].unique())}')


Registros totales: 15,611
Estudiantes únicos: 3,479
Semestres: [np.int64(20221), np.int64(20222), np.int64(20231), np.int64(20232), np.int64(20241), np.int64(20242), np.int64(20251)]


## 5. Historial por estudiante-materia
Se agrupan todos los registros de cada par (Estudiante, Materia) en listas
ordenadas cronológicamente, generando una fila por trayectoria.

In [5]:
hist = (
    df_all
    .groupby(['Documento', 'Materia'])
    .agg(
        Orden_Cronologicos = ('Orden_Cronologico', list),
        Resultados         = ('Resultado',         list),
        Resultados_num     = ('Resultado_num',      list),
        N_registros        = ('Resultado',         'count'),
    )
    .reset_index()
)

print(f'Trayectorias (estudiante x materia): {len(hist):,}')
hist.head(3)


Trayectorias (estudiante x materia): 10,509


,Documento,Materia,Orden_Cronologicos,Resultados,Resultados_num,N_registros
0,1000034155,000506001,"[20221, 20222, 20231]","[Reprobó, Canceló, Reprobó]","[0, 0, 0]",3
1,1000034155,Calculo Diferencial,"[20221, 20222, 20231]","[Canceló, Reprobó, Reprobó Inasistencia]","[0, 0, 0]",3
2,1000083788,000506001,[20222],[Aprobó],[1],1


## 6. Cálculo de la permanencia inicial
Se define una primera versión de la variable `Permanencia` con tres valores:
- **0** → no persistente: solo resultados negativos y sin registros recientes
- **1** → persistente: aprobó al menos una materia en algún semestre
- **2** → pendiente: sus registros llegan al semestre más reciente del dataset
  (imposible determinar si persiste o deserta aún)

In [6]:
df_all['Resultado_num'] = pd.to_numeric(df_all['Resultado_num'], errors='coerce')
MAX_SEMESTRE = int(df_all['Orden_Cronologico'].max())

ultimo_sem = (
    df_all.groupby('Documento')['Orden_Cronologico']
    .max().reset_index()
    .rename(columns={'Orden_Cronologico': 'UltimoOrden_Cronologico'})
)

aprobados = (
    df_all.groupby('Documento')['Resultado_num']
    .max().reset_index()
    .rename(columns={'Resultado_num': 'AlgunaVezAprobo'})
)

ultimo_res = (
    df_all.groupby('Documento')['Resultado_num']
    .last().reset_index()
    .rename(columns={'Resultado_num': 'UltimoResultado'})
)

permanencia = ultimo_sem.merge(aprobados, on='Documento').merge(ultimo_res, on='Documento')


def calcular_perm(row) -> int:
    """Regla de clasificación de permanencia inicial (3 categorías).
    
    CORRECCIÓN: Se verifica primero si el estudiante tiene registros en el
    semestre más reciente (MAX_SEMESTRE). Si es así, se clasifica como
    Pendiente (2) independientemente de si alguna vez aprobó, ya que no es
    posible determinar su desenlace definitivo. Solo después se evalúa si
    aprobó alguna materia (Persistente=1) o no (Desertor=0).
    """
    if row['UltimoOrden_Cronologico'] == MAX_SEMESTRE:
        return 2
    if row['AlgunaVezAprobo'] == 1:
        return 1
    return 0


permanencia['Permanencia'] = permanencia.apply(calcular_perm, axis=1)
df_all = df_all.merge(permanencia[['Documento', 'Permanencia']], on='Documento', how='left')

print('Distribución inicial de Permanencia:')
print(permanencia['Permanencia'].value_counts().sort_index())


Distribución inicial de Permanencia:
Permanencia
0     527
1    1626
2    1326
Name: count, dtype: int64


## 7. Funciones de análisis de trayectorias
Estas funciones analizan la **secuencia de resultados** de cada estudiante-materia
para identificar: primer fracaso, recuperación, deserción o estado pendiente.

In [7]:
def order_to_sem_str(order: int) -> str:
    """Convierte un orden cronológico YYYYS al string 'YYYY-S'."""
    try:
        order = int(order)
        return f'{order // 10}-{order % 10}'
    except Exception:
        return None


def next_semester_order(order: int) -> int:
    """Devuelve el orden cronológico del semestre siguiente."""
    try:
        order = int(order)
        year, term = order // 10, order % 10
        return year * 10 + 2 if term == 1 else (year + 1) * 10 + 1
    except Exception:
        return None


def classify_sequence(res_list: list, sem_list: list, max_semestre: int = None) -> dict:
    """
    Analiza la secuencia de resultados de un estudiante en una materia.

    Returns
    -------
    dict con claves:
      FirstFailSem         : semestre del primer fracaso ('YYYY-S') o None
      OutcomeNextSemester  : resultado en el semestre inmediatamente posterior
      OutcomeEventually    : resultado a largo plazo
      SemRecupero          : semestre de recuperación o None
      AttemptsAfterFirstFail: intentos tras el primer fracaso
    """
    # Limpiar y convertir resultados
    res_clean = []
    for r in res_list:
        if pd.isna(r):
            res_clean.append(None)
        else:
            try:
                res_clean.append(int(float(r)))
            except Exception:
                res_clean.append(None)

    sem_orders = [sem_to_order(s) for s in sem_list]

    # Caso sin fracasos
    valid_res = [r for r in res_clean if r is not None]
    if 0 not in valid_res:
        return {
            'FirstFailSem'         : None,
            'OutcomeNextSemester'  : 'Nunca reprobó',
            'OutcomeEventually'    : 'Nunca reprobó',
            'SemRecupero'          : None,
            'AttemptsAfterFirstFail': 0,
        }

    # Primer fracaso
    first_fail_idx   = next(i for i, v in enumerate(res_clean) if v == 0)
    first_fail_order = sem_orders[first_fail_idx]
    first_fail_str   = order_to_sem_str(first_fail_order)
    next_order       = next_semester_order(first_fail_order)

    # Resultado en el semestre siguiente
    next_result, found_next = None, False
    for j in range(first_fail_idx + 1, len(sem_orders)):
        if sem_orders[j] == next_order:
            next_result, found_next = res_clean[j], True
            break

    if found_next:
        if next_result == 1:
            outcome_next = 'Aprobó en el semestre siguiente'
        elif next_result == 0:
            outcome_next = 'Reprobó en el semestre siguiente'
        else:
            outcome_next = 'Apareció en el semestre siguiente (resultado desconocido)'
    else:
        outcome_next = ('Pendiente'
                        if max_semestre and first_fail_order == max_semestre
                        else 'No apareció en el semestre siguiente')

    # Resultado eventual
    posteriores        = res_clean[first_fail_idx + 1:]
    posteriores_orders = sem_orders[first_fail_idx + 1:]

    if not posteriores or all(r is None for r in posteriores):
        outcome_eventual = ('Pendiente'
                            if max_semestre and first_fail_order == max_semestre
                            else 'Reprobó y desertó')
        sem_recupero, attempts = None, 0

    elif 1 in [r for r in posteriores if r is not None]:
        rel_idx      = next(i for i, v in enumerate(posteriores) if v == 1)
        abs_idx      = first_fail_idx + 1 + rel_idx
        sem_recupero = order_to_sem_str(sem_orders[abs_idx])
        outcome_eventual = 'Reprobó y luego aprobó'
        attempts     = rel_idx + 1

    else:
        outcome_eventual = ('Pendiente'
                            if max_semestre and posteriores_orders
                               and posteriores_orders[-1] == max_semestre
                            else 'Reprobó y siguió reprobando')
        sem_recupero = None
        attempts     = sum(1 for r in posteriores if r == 0)

    return {
        'FirstFailSem'          : first_fail_str,
        'OutcomeNextSemester'   : outcome_next,
        'OutcomeEventually'     : outcome_eventual,
        'SemRecupero'           : sem_recupero,
        'AttemptsAfterFirstFail': attempts,
    }


## 8. Generación de trayectorias
Se aplica `classify_sequence` a cada fila del historial y se consolida
el resultado con la variable de Permanencia.

In [8]:
analizados = hist.apply(
    lambda row: pd.Series(
        classify_sequence(
            row['Resultados_num'],
            row['Orden_Cronologicos'],
            max_semestre=MAX_SEMESTRE,
        )
    ),
    axis=1,
)

trayectorias = pd.concat([hist, analizados], axis=1)

# Incorporar Permanencia si aún no está
if 'Permanencia' not in trayectorias.columns:
    trayectorias = trayectorias.merge(
        permanencia[['Documento', 'Permanencia']], on='Documento', how='left'
    )

# Marcar como pendiente (2) las trayectorias con outcome pendiente
mask_pend = (
    trayectorias['OutcomeNextSemester'].eq('Pendiente') |
    trayectorias['OutcomeEventually'].eq('Pendiente')
)
trayectorias.loc[mask_pend, 'Permanencia'] = 2

# Consolidar permanencia por estudiante con regla de prioridad: 0 > 2 > 1
def prioridad_permanencia(valores: pd.Series) -> int:
    """0 = desertor domina; 2 = pendiente domina sobre 1 (persistente)."""
    if 0 in valores.values: return 0
    if 2 in valores.values: return 2
    return 1

perm_final = (
    trayectorias.groupby('Documento')['Permanencia']
    .apply(prioridad_permanencia)
    .reset_index()
    .rename(columns={'Permanencia': 'Permanencia_final'})
)

trayectorias = (
    trayectorias.drop(columns=['Permanencia'])
    .merge(perm_final, on='Documento', how='left')
    .rename(columns={'Permanencia_final': 'Permanencia'})
)

print('Trayectorias generadas:', len(trayectorias))


Trayectorias generadas: 10509


## 9. Corrección final de permanencia
Se aplica una regla de corrección: si un estudiante aprobó **todas** sus
materias en algún momento (aunque con intentos múltiples), se reclasifica
como persistente (1), salvo que su estado sea pendiente (2).

In [9]:
def asegurar_lista(v):
    """Convierte la celda a lista Python si viene como string serializado."""
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except Exception:
            return []
    return v if isinstance(v, list) else []


trayectorias['Resultados_num'] = trayectorias['Resultados_num'].apply(asegurar_lista)

# ¿Aprobó eventualmente cada materia?
trayectorias['Aprobado_eventual'] = trayectorias['Resultados_num'].apply(
    lambda lst: int(any(int(x) == 1 for x in lst))
)

# ¿Aprobó TODAS sus materias?
todas_aprobadas = (
    trayectorias.groupby('Documento')['Aprobado_eventual']
    .all().astype(int)
    .rename('Todas_materias_aprobadas')
)
trayectorias = trayectorias.merge(todas_aprobadas, on='Documento', how='left')

# Guardar valor original para reportar cambios
trayectorias['Permanencia_original'] = trayectorias['Permanencia']

# Corregir permanencia (solo para registros no pendientes)
mask_no_pend = trayectorias['Permanencia'] != 2
trayectorias.loc[mask_no_pend, 'Permanencia'] = (
    trayectorias.loc[mask_no_pend, 'Todas_materias_aprobadas'].astype(int)
)

# Reporte de cambios
before = trayectorias.groupby('Documento')['Permanencia_original'].first()
after  = trayectorias.groupby('Documento')['Permanencia'].first()
n_cambios = (before != after).sum()

print(f'Documentos totales           : {trayectorias["Documento"].nunique():,}')
print(f'Documentos con cambio perm.  : {n_cambios}')
print('\nDistribución final de Permanencia (por estudiante):')
print(after.value_counts().sort_index())


Documentos totales           : 3,479
Documentos con cambio perm.  : 311

Distribución final de Permanencia (por estudiante):
Permanencia
0     838
1    1315
2    1326
Name: count, dtype: int64


## 10. Guardado del dataset analítico
Se exporta el DataFrame `trayectorias` como archivo Excel.
Este archivo es la **entrada del Notebook 3** (modelado analítico).

In [10]:
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
trayectorias.to_excel(RUTA_SALIDA, index=False)

print(f'Dataset analítico guardado:')
print(f'  {RUTA_SALIDA}')
print(f'  {len(trayectorias):,} filas  ×  {len(trayectorias.columns)} columnas')


Dataset analítico guardado:
  /sessions/trusting-laughing-brahmagupta/mnt/Academic Performance, Course Load, and Student Persistence A Longitudinal Modeling of Academic Trajectories in an Engineering Program/Proyecto_Final_STEM_COPIA/Analytical_Dataset_Building/Study_Data/Oficial_Data_ToAnalisis.xlsx
  10,509 filas  ×  15 columnas
